###1) Kontrola środowiska i danych

In [0]:
from pyspark.sql import functions as F

import numpy as np
import pandas as pd
import sklearn

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)


MODEL_DATASET_TABLE = (
    "workspace.energy_gold."
    "market_model_dataset_15m"
)


model_spark_df = (
    spark.table(MODEL_DATASET_TABLE)
    .orderBy("target_interval_start_utc")
)


print(f"Tabela: {MODEL_DATASET_TABLE}")
print(f"Wersja scikit-learn: {sklearn.__version__}")
print(f"Liczba rekordów: {model_spark_df.count()}")
print(f"Liczba kolumn: {len(model_spark_df.columns)}")


display(
    model_spark_df
    .groupBy("dataset_split")
    .agg(
        F.count("*").alias("row_count"),
        F.min("target_interval_start_utc").alias(
            "min_target_time_utc"
        ),
        F.max("target_interval_start_utc").alias(
            "max_target_time_utc"
        )
    )
    .orderBy(
        F.when(
            F.col("dataset_split") == "TRAIN", 1
        )
        .when(
            F.col("dataset_split") == "VALIDATION", 2
        )
        .when(
            F.col("dataset_split") == "TEST", 3
        )
        .otherwise(4)
    )
)

###2) Konwersja do Pandas i wydzielenie zbiorów

In [0]:
model_pdf = (
    model_spark_df
    .toPandas()
    .sort_values(
        "target_interval_start_utc"
    )
    .reset_index(drop=True)
)


train_pdf = (
    model_pdf[
        model_pdf["dataset_split"] == "TRAIN"
    ]
    .copy()
)


validation_pdf = (
    model_pdf[
        model_pdf["dataset_split"] == "VALIDATION"
    ]
    .copy()
)


test_pdf = (
    model_pdf[
        model_pdf["dataset_split"] == "TEST"
    ]
    .copy()
)


print(f"TRAIN: {len(train_pdf)}")
print(f"VALIDATION: {len(validation_pdf)}")
print(f"TEST: {len(test_pdf)}")
print(f"RAZEM: {len(model_pdf)}")


split_sizes = {
    "TRAIN": len(train_pdf),
    "VALIDATION": len(validation_pdf),
    "TEST": len(test_pdf),
}

if any(size <= 0 for size in split_sizes.values()):
    raise RuntimeError(
        f"Co najmniej jeden split jest pusty: {split_sizes}"
    )

if sum(split_sizes.values()) != len(model_pdf):
    raise RuntimeError("Splity nie obejmują całego zbioru modelowego.")

if not (
    train_pdf["target_interval_start_utc"].max()
    < validation_pdf["target_interval_start_utc"].min()
    < validation_pdf["target_interval_start_utc"].max()
    < test_pdf["target_interval_start_utc"].min()
):
    raise RuntimeError("Splity nie są uporządkowane chronologicznie.")


print()
print(
    "Konwersja do Pandas i kontrola "
    "chronologii zostały zaliczone."
)

###3) Funkcja metryk i baseline'y

In [0]:
def calculate_regression_metrics(
    actual_values,
    predicted_values,
):
    actual_array = np.asarray(
        actual_values,
        dtype=float
    )

    predicted_array = np.asarray(
        predicted_values,
        dtype=float
    )

    return {
        "mae": float(
            mean_absolute_error(
                actual_array,
                predicted_array
            )
        ),
        "rmse": float(
            np.sqrt(
                mean_squared_error(
                    actual_array,
                    predicted_array
                )
            )
        ),
        "r2": float(
            r2_score(
                actual_array,
                predicted_array
            )
        ),
        "mean_error": float(
            np.mean(
                predicted_array - actual_array
            )
        ),
    }


baseline_results = []


def add_baseline_result(
    target_name,
    model_name,
    actual_values,
    predicted_values,
):
    metrics = calculate_regression_metrics(
        actual_values,
        predicted_values
    )

    baseline_results.append({
        "target": target_name,
        "model_name": model_name,
        "dataset_split": "VALIDATION",
        **metrics,
    })


# Baseline produkcji PV
add_baseline_result(
    target_name="PV_POWER_KW",
    model_name="LAST_OBSERVED_POWER",
    actual_values=validation_pdf[
        "actual_pv_power_kw"
    ],
    predicted_values=validation_pdf[
        "observed_ac_power_kw"
    ],
)


# Baseline ceny: DAM
add_baseline_result(
    target_name="FMM_PRICE_USD_MWH",
    model_name="DAM_PRICE",
    actual_values=validation_pdf[
        "actual_fmm_lmp_usd_mwh"
    ],
    predicted_values=validation_pdf[
        "dam_lmp_usd_mwh"
    ],
)


# Baseline ceny: ostatnia znana FMM
add_baseline_result(
    target_name="FMM_PRICE_USD_MWH",
    model_name="LAST_KNOWN_FMM_PRICE",
    actual_values=validation_pdf[
        "actual_fmm_lmp_usd_mwh"
    ],
    predicted_values=validation_pdf[
        "last_known_fmm_lmp_usd_mwh"
    ],
)


# Baseline ceny: średnia z 4 ostatnich FMM
add_baseline_result(
    target_name="FMM_PRICE_USD_MWH",
    model_name="MEAN_LAST_4_FMM_PRICES",
    actual_values=validation_pdf[
        "actual_fmm_lmp_usd_mwh"
    ],
    predicted_values=validation_pdf[
        "mean_last_4_known_fmm_lmp_usd_mwh"
    ],
)


baseline_results_pdf = (
    pd.DataFrame(baseline_results)
    .sort_values(
        ["target", "mae"]
    )
    .reset_index(drop=True)
)


display(baseline_results_pdf)

###4) Cechy PV i ceny

In [0]:
pv_feature_columns = [
    "decision_horizon_minutes",
    "observed_ac_power_kw",
    "observed_pv_energy_mwh",
    "observed_poa_wm2",
    "observed_temperature_c",
    "observed_wind_speed_ms",
    "forecast_temperature_c",
    "forecast_relative_humidity_pct",
    "forecast_cloud_cover_pct",
    "forecast_wind_speed_ms",
    "forecast_shortwave_radiation_wm2",
    "forecast_gti_wm2",
    "forecast_is_day",
    "weather_forecast_age_minutes",
    "weather_lead_time_hours",
    "target_quarter",
    "target_time_of_day_sin",
    "target_time_of_day_cos",
    "target_day_of_week_sin",
    "target_day_of_week_cos",
    "observed_wind_direction_sin",
    "observed_wind_direction_cos",
    "forecast_wind_direction_sin",
    "forecast_wind_direction_cos",
]


price_feature_columns = [
    "decision_horizon_minutes",
    "observed_ac_power_kw",
    "observed_poa_wm2",
    "forecast_temperature_c",
    "forecast_relative_humidity_pct",
    "forecast_cloud_cover_pct",
    "forecast_wind_speed_ms",
    "forecast_shortwave_radiation_wm2",
    "forecast_gti_wm2",
    "forecast_is_day",
    "dam_lmp_usd_mwh",
    "dam_energy_component_usd_mwh",
    "dam_congestion_component_usd_mwh",
    "dam_loss_component_usd_mwh",
    "last_known_fmm_lmp_usd_mwh",
    "mean_last_4_known_fmm_lmp_usd_mwh",
    "min_last_4_known_fmm_lmp_usd_mwh",
    "max_last_4_known_fmm_lmp_usd_mwh",
    "last_known_fmm_to_target_gap_minutes",
    "target_quarter",
    "target_time_of_day_sin",
    "target_time_of_day_cos",
    "target_day_of_week_sin",
    "target_day_of_week_cos",
    "last_known_fmm_minus_dam_usd_mwh",
    "mean_last_4_fmm_minus_dam_usd_mwh",
    "last_4_known_fmm_range_usd_mwh",
]


pv_target_column = "actual_pv_power_kw"
price_target_column = "actual_fmm_lmp_usd_mwh"


required_ml_columns = set(
    pv_feature_columns
    + price_feature_columns
    + [
        pv_target_column,
        price_target_column,
    ]
)


missing_ml_columns = sorted(
    required_ml_columns
    - set(model_pdf.columns)
)


future_columns_in_features = sorted(
    {
        column_name
        for column_name in (
            pv_feature_columns
            + price_feature_columns
        )
        if column_name.startswith("actual_")
    }
)


if missing_ml_columns:
    raise RuntimeError(
        "Brakuje kolumn ML: "
        f"{missing_ml_columns}"
    )


if future_columns_in_features:
    raise RuntimeError(
        "W cechach znaleziono przyszłe dane: "
        f"{future_columns_in_features}"
    )


X_train_pv = train_pdf[
    pv_feature_columns
].astype(float)

X_validation_pv = validation_pdf[
    pv_feature_columns
].astype(float)

y_train_pv = train_pdf[
    pv_target_column
].astype(float)

y_validation_pv = validation_pdf[
    pv_target_column
].astype(float)


X_train_price = train_pdf[
    price_feature_columns
].astype(float)

X_validation_price = validation_pdf[
    price_feature_columns
].astype(float)

y_train_price = train_pdf[
    price_target_column
].astype(float)

y_validation_price = validation_pdf[
    price_target_column
].astype(float)


assert not X_train_pv.isna().any().any()
assert not X_validation_pv.isna().any().any()
assert not X_train_price.isna().any().any()
assert not X_validation_price.isna().any().any()


print(f"Liczba cech PV: {len(pv_feature_columns)}")
print(
    "Rozmiar TRAIN PV: "
    f"{X_train_pv.shape}"
)
print(
    "Rozmiar VALIDATION PV: "
    f"{X_validation_pv.shape}"
)

print()

print(
    "Liczba cech ceny: "
    f"{len(price_feature_columns)}"
)
print(
    "Rozmiar TRAIN ceny: "
    f"{X_train_price.shape}"
)
print(
    "Rozmiar VALIDATION ceny: "
    f"{X_validation_price.shape}"
)

print()
print("Kontrola cech ML została zaliczona.")

###5) Trenowanie modeli kandydackich

In [0]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import (
    HistGradientBoostingRegressor,
)


price_market_feature_columns = [
    "decision_horizon_minutes",
    "dam_lmp_usd_mwh",
    "dam_energy_component_usd_mwh",
    "dam_congestion_component_usd_mwh",
    "dam_loss_component_usd_mwh",
    "last_known_fmm_lmp_usd_mwh",
    "mean_last_4_known_fmm_lmp_usd_mwh",
    "min_last_4_known_fmm_lmp_usd_mwh",
    "max_last_4_known_fmm_lmp_usd_mwh",
    "last_known_fmm_to_target_gap_minutes",
    "target_quarter",
    "target_time_of_day_sin",
    "target_time_of_day_cos",
    "target_day_of_week_sin",
    "target_day_of_week_cos",
    "last_known_fmm_minus_dam_usd_mwh",
    "mean_last_4_fmm_minus_dam_usd_mwh",
    "last_4_known_fmm_range_usd_mwh",
]


price_weather_context_columns = [
    "observed_ac_power_kw",
    "observed_poa_wm2",
    "forecast_temperature_c",
    "forecast_relative_humidity_pct",
    "forecast_cloud_cover_pct",
    "forecast_wind_speed_ms",
    "forecast_shortwave_radiation_wm2",
    "forecast_gti_wm2",
    "forecast_is_day",
]


price_market_weather_feature_columns = (
    price_market_feature_columns
    + price_weather_context_columns
)


if (
    set(price_market_weather_feature_columns)
    != set(price_feature_columns)
):
    raise RuntimeError(
        "Podział cech ceny nie odpowiada "
        "wcześniejszej liście."
    )


X_train_price_market = train_pdf[
    price_market_feature_columns
].astype(float)

X_validation_price_market = validation_pdf[
    price_market_feature_columns
].astype(float)

X_train_price_market_weather = train_pdf[
    price_market_weather_feature_columns
].astype(float)

X_validation_price_market_weather = validation_pdf[
    price_market_weather_feature_columns
].astype(float)


candidate_models = {
    "PV_RIDGE": {
        "target": "PV_POWER_KW",
        "model": Pipeline([
            ("scaler", StandardScaler()),
            (
                "regressor",
                Ridge(alpha=10.0)
            ),
        ]),
        "features": pv_feature_columns,
        "X_train": X_train_pv,
        "y_train": y_train_pv,
        "X_validation": X_validation_pv,
        "y_validation": y_validation_pv,
        "clip_min": 0.0,
        "clip_max": 893.0,
    },

    "PV_HGB": {
        "target": "PV_POWER_KW",
        "model": HistGradientBoostingRegressor(
            loss="squared_error",
            learning_rate=0.05,
            max_iter=250,
            max_leaf_nodes=15,
            min_samples_leaf=20,
            l2_regularization=1.0,
            random_state=42,
        ),
        "features": pv_feature_columns,
        "X_train": X_train_pv,
        "y_train": y_train_pv,
        "X_validation": X_validation_pv,
        "y_validation": y_validation_pv,
        "clip_min": 0.0,
        "clip_max": 893.0,
    },

    "PRICE_RIDGE_MARKET_ONLY": {
        "target": "FMM_PRICE_USD_MWH",
        "model": Pipeline([
            ("scaler", StandardScaler()),
            (
                "regressor",
                Ridge(alpha=10.0)
            ),
        ]),
        "features": price_market_feature_columns,
        "X_train": X_train_price_market,
        "y_train": y_train_price,
        "X_validation": X_validation_price_market,
        "y_validation": y_validation_price,
        "clip_min": None,
        "clip_max": None,
    },

    "PRICE_HGB_MARKET_ONLY": {
        "target": "FMM_PRICE_USD_MWH",
        "model": HistGradientBoostingRegressor(
            loss="squared_error",
            learning_rate=0.05,
            max_iter=250,
            max_leaf_nodes=15,
            min_samples_leaf=30,
            l2_regularization=5.0,
            random_state=42,
        ),
        "features": price_market_feature_columns,
        "X_train": X_train_price_market,
        "y_train": y_train_price,
        "X_validation": X_validation_price_market,
        "y_validation": y_validation_price,
        "clip_min": None,
        "clip_max": None,
    },

    "PRICE_RIDGE_MARKET_WEATHER": {
        "target": "FMM_PRICE_USD_MWH",
        "model": Pipeline([
            ("scaler", StandardScaler()),
            (
                "regressor",
                Ridge(alpha=10.0)
            ),
        ]),
        "features": (
            price_market_weather_feature_columns
        ),
        "X_train": X_train_price_market_weather,
        "y_train": y_train_price,
        "X_validation": (
            X_validation_price_market_weather
        ),
        "y_validation": y_validation_price,
        "clip_min": None,
        "clip_max": None,
    },

    "PRICE_HGB_MARKET_WEATHER": {
        "target": "FMM_PRICE_USD_MWH",
        "model": HistGradientBoostingRegressor(
            loss="squared_error",
            learning_rate=0.05,
            max_iter=250,
            max_leaf_nodes=15,
            min_samples_leaf=30,
            l2_regularization=5.0,
            random_state=42,
        ),
        "features": (
            price_market_weather_feature_columns
        ),
        "X_train": X_train_price_market_weather,
        "y_train": y_train_price,
        "X_validation": (
            X_validation_price_market_weather
        ),
        "y_validation": y_validation_price,
        "clip_min": None,
        "clip_max": None,
    },
}


trained_candidate_models = {}
validation_predictions = {}
candidate_results = []


for model_name, configuration in (
    candidate_models.items()
):
    estimator = configuration["model"]

    estimator.fit(
        configuration["X_train"],
        configuration["y_train"]
    )

    prediction = estimator.predict(
        configuration["X_validation"]
    )

    if configuration["clip_min"] is not None:
        prediction = np.maximum(
            prediction,
            configuration["clip_min"]
        )

    if configuration["clip_max"] is not None:
        prediction = np.minimum(
            prediction,
            configuration["clip_max"]
        )

    metrics = calculate_regression_metrics(
        configuration["y_validation"],
        prediction
    )

    trained_candidate_models[model_name] = {
        "model": estimator,
        "features": configuration["features"],
        "target": configuration["target"],
    }

    validation_predictions[model_name] = prediction

    candidate_results.append({
        "target": configuration["target"],
        "model_name": model_name,
        "dataset_split": "VALIDATION",
        "feature_count": len(
            configuration["features"]
        ),
        **metrics,
    })


candidate_results_pdf = pd.DataFrame(
    candidate_results
)


all_validation_results_pdf = (
    pd.concat(
        [
            baseline_results_pdf.assign(
                feature_count=np.nan
            ),
            candidate_results_pdf,
        ],
        ignore_index=True
    )
    .sort_values(
        ["target", "mae"]
    )
    .reset_index(drop=True)
)


display(all_validation_results_pdf)

###6) Finalne uczenie i ocena TEST

In [0]:
from sklearn.base import clone


# Po wyborze modelu łączymy TRAIN i VALIDATION.
train_validation_pdf = (
    pd.concat(
        [train_pdf, validation_pdf],
        ignore_index=True
    )
    .sort_values(
        "target_interval_start_utc"
    )
    .reset_index(drop=True)
)


X_train_validation_pv = (
    train_validation_pdf[
        pv_feature_columns
    ]
    .astype(float)
)

y_train_validation_pv = (
    train_validation_pdf[
        pv_target_column
    ]
    .astype(float)
)

X_test_pv = (
    test_pdf[
        pv_feature_columns
    ]
    .astype(float)
)

y_test_pv = (
    test_pdf[
        pv_target_column
    ]
    .astype(float)
)


# Ponowne trenowanie wybranego PV_HGB
final_pv_model = clone(
    candidate_models["PV_HGB"]["model"]
)

final_pv_model.fit(
    X_train_validation_pv,
    y_train_validation_pv
)


predicted_test_pv_kw = (
    final_pv_model.predict(X_test_pv)
)

predicted_test_pv_kw = np.clip(
    predicted_test_pv_kw,
    0.0,
    893.0
)


# Wybrana prognoza ceny to znana cena DAM.
predicted_test_fmm_price = (
    test_pdf["dam_lmp_usd_mwh"]
    .astype(float)
    .to_numpy()
)


test_model_results = []


def add_test_result(
    target_name,
    model_name,
    actual_values,
    predicted_values,
):
    metrics = calculate_regression_metrics(
        actual_values,
        predicted_values
    )

    test_model_results.append({
        "target": target_name,
        "model_name": model_name,
        "dataset_split": "TEST",
        **metrics,
    })


# Finalny model PV
add_test_result(
    target_name="PV_POWER_KW",
    model_name="PV_HGB_FINAL",
    actual_values=y_test_pv,
    predicted_values=predicted_test_pv_kw,
)


# Baseline PV do porównania
add_test_result(
    target_name="PV_POWER_KW",
    model_name="LAST_OBSERVED_POWER",
    actual_values=y_test_pv,
    predicted_values=test_pdf[
        "observed_ac_power_kw"
    ],
)


# Finalna metoda prognozy ceny
add_test_result(
    target_name="FMM_PRICE_USD_MWH",
    model_name="DAM_PRICE_FINAL",
    actual_values=test_pdf[
        "actual_fmm_lmp_usd_mwh"
    ],
    predicted_values=predicted_test_fmm_price,
)


# Pozostałe baseline’y ceny tylko do porównania
add_test_result(
    target_name="FMM_PRICE_USD_MWH",
    model_name="LAST_KNOWN_FMM_PRICE",
    actual_values=test_pdf[
        "actual_fmm_lmp_usd_mwh"
    ],
    predicted_values=test_pdf[
        "last_known_fmm_lmp_usd_mwh"
    ],
)

add_test_result(
    target_name="FMM_PRICE_USD_MWH",
    model_name="MEAN_LAST_4_FMM_PRICES",
    actual_values=test_pdf[
        "actual_fmm_lmp_usd_mwh"
    ],
    predicted_values=test_pdf[
        "mean_last_4_known_fmm_lmp_usd_mwh"
    ],
)


test_model_results_pdf = (
    pd.DataFrame(test_model_results)
    .sort_values(["target", "mae"])
    .reset_index(drop=True)
)


display(test_model_results_pdf)


# Predykcje potrzebne w późniejszym backteście.
test_predictions_pdf = (
    test_pdf[
        [
            "decision_id",
            "decision_time_utc",
            "target_interval_start_utc",
            "target_interval_end_utc",
            "target_time_local",
            "dataset_split",
            "actual_pv_power_kw",
            "actual_pv_energy_mwh",
            "actual_fmm_lmp_usd_mwh",
            "dam_lmp_usd_mwh",
        ]
    ]
    .copy()
)


test_predictions_pdf[
    "predicted_pv_power_kw"
] = predicted_test_pv_kw

test_predictions_pdf[
    "predicted_pv_energy_mwh"
] = (
    predicted_test_pv_kw
    * 0.25
    / 1000.0
)

test_predictions_pdf[
    "predicted_fmm_lmp_usd_mwh"
] = predicted_test_fmm_price

test_predictions_pdf[
    "pv_model_name"
] = "PV_HGB_FINAL"

test_predictions_pdf[
    "price_model_name"
] = "DAM_PRICE_FINAL"


if len(test_predictions_pdf) != len(test_pdf):
    raise RuntimeError(
        "Liczba predykcji TEST nie zgadza się z liczbą rekordów TEST."
    )
assert not test_predictions_pdf[
    "predicted_pv_power_kw"
].isna().any()
assert not test_predictions_pdf[
    "predicted_fmm_lmp_usd_mwh"
].isna().any()


print()
print(
    "Finalna ocena TEST została wykonana "
    "jednokrotnie."
)
print(
    "Liczba predykcji do backtestu: "
    f"{len(test_predictions_pdf)}"
)

###7) Zapis predykcji TEST do Delta

In [0]:
from pyspark.sql import functions as F

TEST_PREDICTIONS_TABLE = (
    "workspace.energy_gold."
    "market_model_test_predictions_15m"
)

test_predictions_spark_df = (
    spark.createDataFrame(test_predictions_pdf)
    .withColumn(
        "pv_model_name",
        F.lit("PV_HGB_FINAL")
    )
    .withColumn(
        "price_forecast_method",
        F.lit("DAM_PRICE_FINAL")
    )
    .withColumn(
        "_model_version",
        F.lit("v1")
    )
    .withColumn(
        "_predictions_created_at_utc",
        F.current_timestamp()
    )
)

(
    test_predictions_spark_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TEST_PREDICTIONS_TABLE)
)

saved_predictions_df = spark.table(
    TEST_PREDICTIONS_TABLE
)

print(f"Zapisano tabelę: {TEST_PREDICTIONS_TABLE}")
print(f"Liczba rekordów: {saved_predictions_df.count()}")

display(
    saved_predictions_df
    .orderBy("target_interval_start_utc")
    .limit(10)
)

###8) Zapis do MLflow

In [0]:
import mlflow
import mlflow.sklearn
import numpy as np

from mlflow.models import infer_signature
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

predictions_pdf = (
    spark.table(
        "workspace.energy_gold."
        "market_model_test_predictions_15m"
    )
    .orderBy("target_interval_start_utc")
    .toPandas()
)

if predictions_pdf.empty:
    raise RuntimeError("Brak predykcji TEST do zapisania w MLflow.")

pv_actual = predictions_pdf["actual_pv_power_kw"]
pv_predicted = predictions_pdf["predicted_pv_power_kw"]
price_actual = predictions_pdf["actual_fmm_lmp_usd_mwh"]
price_predicted = predictions_pdf["predicted_fmm_lmp_usd_mwh"]

metrics = {
    "test_pv_mae_kw": mean_absolute_error(
        pv_actual, pv_predicted
    ),
    "test_pv_rmse_kw": np.sqrt(
        mean_squared_error(pv_actual, pv_predicted)
    ),
    "test_pv_r2": r2_score(pv_actual, pv_predicted),
    "test_price_mae_usd_mwh": mean_absolute_error(
        price_actual, price_predicted
    ),
    "test_price_rmse_usd_mwh": np.sqrt(
        mean_squared_error(price_actual, price_predicted)
    ),
    "test_price_r2": r2_score(
        price_actual, price_predicted
    ),
}

feature_names = list(final_pv_model.feature_names_in_)
input_example = (
    test_pdf[feature_names]
    .head(5)
    .astype(float)
    .copy()
)
example_predictions = final_pv_model.predict(input_example)
model_signature = infer_signature(
    input_example,
    example_predictions,
)

test_period_start = str(
    predictions_pdf["target_interval_start_utc"].min()
)
test_period_end = str(
    predictions_pdf["target_interval_end_utc"].max()
)

if mlflow.active_run() is not None:
    mlflow.end_run()

with mlflow.start_run(
    run_name="energy_market_replay_final_parameterized"
) as run:
    mlflow.log_params({
        "pv_model": "PV_HGB_FINAL",
        "price_forecast": "DAM_PRICE_FINAL",
        "dataset_split": "TEST",
        "test_row_count": len(predictions_pdf),
        "evaluation_period_start": test_period_start,
        "evaluation_period_end": test_period_end,
        "prediction_interval_minutes": 15,
    })
    mlflow.log_metrics({
        metric_name: float(metric_value)
        for metric_name, metric_value in metrics.items()
    })
    mlflow.set_tags({
        "project": "energy_market_replay",
        "source_systems": "NLR_OPEN_METEO_CAISO",
        "pipeline_version": "parameterized_v3",
    })
    signed_model_info = mlflow.sklearn.log_model(
        sk_model=final_pv_model,
        name="pv_hgb_final_signed",
        input_example=input_example,
        signature=model_signature,
    )
    run_id = run.info.run_id

print(f"Zapisano run MLflow: {run_id}")
print(f"Model URI: {signed_model_info.model_uri}")
for metric_name, metric_value in metrics.items():
    print(f"{metric_name}: {metric_value:.6f}")

###8a) Kontrola sygnatury modelu zapisanego w bieżącym runie

In [0]:
if "signed_model_info" not in globals():
    raise NameError(
        "Brak modelu z bieżącego runu. Uruchom notebook od początku."
    )

print("Model został zapisany z sygnaturą.")
print(f"Model URI: {signed_model_info.model_uri}")
print(model_signature)

###9) Rejestracja modelu w Unity Catalog

In [0]:
import mlflow

mlflow.set_registry_uri("databricks-uc")

if "signed_model_info" not in globals():
    raise NameError(
        "Brak modelu z bieżącego runu. Uruchom notebook od początku."
    )

REGISTERED_MODEL_NAME = (
    "workspace.energy_gold.pv_power_hgb_15m"
)

registered_version = mlflow.register_model(
    model_uri=signed_model_info.model_uri,
    name=REGISTERED_MODEL_NAME,
)

print(f"Model: {REGISTERED_MODEL_NAME}")
print(f"Wersja: {registered_version.version}")
print(f"Status: {registered_version.status}")

###10) Alias i test odczytu modelu

In [0]:
import mlflow
import numpy as np

from mlflow import MlflowClient

mlflow.set_registry_uri("databricks-uc")

MODEL_NAME = "workspace.energy_gold.pv_power_hgb_15m"

if "registered_version" not in globals():
    raise NameError(
        "Brak zarejestrowanej wersji. Uruchom notebook od początku."
    )

client = MlflowClient(registry_uri="databricks-uc")
client.set_registered_model_alias(
    name=MODEL_NAME,
    alias="Champion",
    version=str(registered_version.version),
)

loaded_model = mlflow.pyfunc.load_model(
    f"models:/{MODEL_NAME}@Champion"
)

feature_names = [
    column.name
    for column in loaded_model.metadata.get_input_schema().inputs
]

sample_pdf = (
    spark.table(
        "workspace.energy_gold.market_model_dataset_15m"
    )
    .filter("dataset_split = 'TEST'")
    .select(*feature_names)
    .limit(5)
    .toPandas()
)

sample_predictions = np.maximum(
    loaded_model.predict(sample_pdf),
    0.0,
)

print("Alias: Champion")
print(f"Wersja: {registered_version.version}")
print(f"Liczba cech: {len(feature_names)}")
print(f"Predykcje testowe: {sample_predictions}")